In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

# ==========================================
# 1. LOAD DATASET & TRAIN MODEL
# ==========================================

print("🤖 Initializing Machine Learning Engine...")

df = pd.read_csv(r"D:\mini project\flight price prediction\Clean_Dataset.csv\Clean_Dataset.csv")

if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

categorical_cols = X_train.select_dtypes(
    include=["object", "string"]
).columns

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_encoded = X_train.copy()

X_train_encoded[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols]
)

model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

model.fit(X_train_encoded, y_train)

print("Engine Ready.\n")

# ==========================================
# 2. DISPLAY AVAILABLE CITIES
# ==========================================

print("=== MULTI-AIRLINE PRICE COMPARISON ===")

available_sources = sorted(
    df["source_city"].unique()
)

available_destinations = sorted(
    df["destination_city"].unique()
)

print(f"Available Origins:      {available_sources}")
print(f"Available Destinations: {available_destinations}")
print("-" * 65)

# ==========================================
# 3. TAKE USER INPUT
# ==========================================

user_source = input(
    "\nEnter origin city (FROM): "
).strip()

user_dest = input(
    "Enter destination city (TO): "
).strip()

# ==========================================
# 4. FILTER SELECTED ROUTE
# ==========================================

route_matches = df[
    (df["source_city"].str.lower() == user_source.lower())
    & (df["destination_city"].str.lower() == user_dest.lower())
]

if route_matches.empty:

    print(
        f"\n❌ No flight paths found from "
        f"'{user_source}' to '{user_dest}'."
    )

else:

    user_class = input(
        "Select Class (Economy/Business): "
    ).strip()

    user_days = int(
        input(
            "Enter days remaining before departure: "
        )
    )

    # ==========================================
    # 5. FIND AVAILABLE AIRLINES
    # ==========================================

    unique_airlines = route_matches[
        "airline"
    ].unique()

    report_data = []

    print(
        f"\n🔍 Analyzing {len(unique_airlines)} "
        "airlines on this route..."
    )

    # ==========================================
    # 6. PREDICT PRICE FOR EACH AIRLINE
    # ==========================================

    for airline in unique_airlines:

        airline_matches = route_matches[
            (route_matches["airline"] == airline)
            & (
                route_matches["class"].str.lower()
                == user_class.lower()
            )
        ]

        if airline_matches.empty:
            continue

        historical_avg = airline_matches[
            "price"
        ].mean()

        inferred_duration = airline_matches[
            "duration"
        ].median()

        inferred_dep_time = airline_matches[
            "departure_time"
        ].mode()[0]

        inferred_stops = airline_matches[
            "stops"
        ].mode()[0]

        inferred_arr_time = (
            airline_matches["arrival_time"].mode()[0]
            if "arrival_time" in df.columns
            else "Evening"
        )

        inferred_flight = (
            airline_matches["flight"].mode()[0]
            if "flight" in df.columns
            else "AI-101"
        )

        # ==========================================
        # 7. CREATE MODEL INPUT
        # ==========================================

        raw_input = {
            "airline": airline,
            "flight": inferred_flight,
            "source_city": user_source,
            "departure_time": inferred_dep_time,
            "stops": inferred_stops,
            "arrival_time": inferred_arr_time,
            "destination_city": user_dest,
            "class": user_class,
            "duration": inferred_duration,
            "days_left": user_days
        }

        # Match input values with dataset categories
        for col in categorical_cols:

            if col in raw_input:

                for category in X[col].unique():

                    if (
                        str(category).lower()
                        == str(raw_input[col]).lower()
                    ):
                        raw_input[col] = category
                        break

        # ==========================================
        # 8. ENCODE USER INPUT
        # ==========================================

        user_frame = pd.DataFrame(
            [raw_input]
        )[X.columns]

        user_frame_encoded = user_frame.copy()

        user_frame_encoded[categorical_cols] = (
            encoder.transform(
                user_frame[categorical_cols]
            )
        )

        # ==========================================
        # 9. PREDICT FLIGHT PRICE
        # ==========================================

        ml_prediction = model.predict(
            user_frame_encoded
        )[0]

        # ==========================================
        # 10. STORE RESULT
        # ==========================================

        report_data.append({
            "Airline": airline,
            "Duration": f"{inferred_duration:.1f} hrs",
            "Stops": inferred_stops,
            "Dataset Avg Price": f"₹{historical_avg:,.2f}",
            "Model Prediction": f"₹{ml_prediction:,.2f}"
        })

    # ==========================================
    # 11. DISPLAY FINAL REPORT
    # ==========================================

    comparison_df = pd.DataFrame(
        report_data
    )

    print("\n" + "=" * 75)

    print(
        f"📊 FARE COMPARISON REPORT: "
        f"{user_source.upper()} ➔ "
        f"{user_dest.upper()} "
        f"({user_class.capitalize()})"
    )

    print(
        f"   Booking assumed "
        f"{user_days} days before departure."
    )

    print("=" * 75)

    if comparison_df.empty:

        print(
            f"No airlines found matching "
            f"the class '{user_class}' "
            f"on this route."
        )

    else:

        print(
            comparison_df.to_string(
                index=False
            )
        )

    print("=" * 75)

🤖 Initializing Machine Learning Engine...


C:\Users\Dell\AppData\Local\Temp\ipykernel_17064\2806671277.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns # FIXED HERE


Engine Ready.

=== MULTI-AIRLINE PRICE COMPARISON ===
Available Origins: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
Available Destinations: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
-----------------------------------------------------------------
